In [ ]:
from private_path_query_utils import Vertex, Edge, EdgeState, Graph, GraphPath
from private_path_query_logic import build_bob_q, ordered_symbols, directed_pairs_from_edges

# Build a small directed-edge list by setting adjacency entries.
# States: 0=no edge, 1=blocked, 2=traversable
edge_states = [
    [0, 2, 0, 0],
    [0, 0, 1, 2],
    [0, 0, 0, 2],
    [0, 0, 0, 0],
]

vertices = [Vertex(i) for i in range(4)]
graph = Graph(vertices=vertices, edge_states=edge_states)
edges = graph.to_directed_edges()

print("Directed edges (u -> v, state):")
for e in edges:
    print(f"  {e.vertex1.id} -> {e.vertex2.id}, state={int(e.state)}")


Directed edges (u -> v, state):
  0 -> 1, state=2
  1 -> 2, state=1
  1 -> 3, state=2
  2 -> 3, state=2


In [ ]:
# Graph-plan style route: 0 -> 1 -> 3
plan_edges = [
    Edge(vertices[0], vertices[1], EdgeState.TRAVERSABLE),
    Edge(vertices[1], vertices[3], EdgeState.TRAVERSABLE),
]
plan = GraphPath(start=vertices[0], moves=plan_edges)

print("\nGraphPath (edge sequence):")
for step, e in enumerate(plan.moves):
    print(f"  step {step}: {e.vertex1.id} -> {e.vertex2.id}")

T = len(plan.moves)
V = len(vertices)

# Optional compact domain from edges present in graph.
compact_pairs = directed_pairs_from_edges(edges, V)
syms = ordered_symbols(T=T, V=V, directed_pairs=compact_pairs)

# This now prints CNF clauses generated from Bob physics.
q_bob, w_bob = build_bob_q(
    edges=edges,
    T=T,
    V=V,
    symbols=syms,
    directed_pairs=compact_pairs,
    print_cnf_clauses=True,
)

print("\nQ_bob shape:", q_bob.shape)
print("w_bob shape:", w_bob.shape)
print("First 5 rows of Q_bob:")
print(q_bob[:5])
print("First 5 bob clause weights:", w_bob[:5])


GraphPath (edge sequence):
  step 0: 0 -> 1
  step 1: 1 -> 3
=== build_bob_q: Bob physics CNF clauses ===
bob_physics[0] = Active(0, 1)
  cnf[0] (from local 0, weight=100.0) = Active(0, 1)
bob_physics[1] = Allowed(0, 1)
  cnf[1] (from local 0, weight=100.0) = Allowed(0, 1)
bob_physics[2] = (Allowed(0, 1) ==> Active(0, 1))
  cnf[2] (from local 0, weight=100.0) = (Active(0, 1) | ~Allowed(0, 1))
bob_physics[3] = Active(1, 2)
  cnf[3] (from local 0, weight=100.0) = Active(1, 2)
bob_physics[4] = ~Allowed(1, 2)
  cnf[4] (from local 0, weight=1.0) = ~Allowed(1, 2)
bob_physics[5] = (Allowed(1, 2) ==> Active(1, 2))
  cnf[5] (from local 0, weight=100.0) = (Active(1, 2) | ~Allowed(1, 2))
bob_physics[6] = Active(1, 3)
  cnf[6] (from local 0, weight=100.0) = Active(1, 3)
bob_physics[7] = Allowed(1, 3)
  cnf[7] (from local 0, weight=100.0) = Allowed(1, 3)
bob_physics[8] = (Allowed(1, 3) ==> Active(1, 3))
  cnf[8] (from local 0, weight=100.0) = (Active(1, 3) | ~Allowed(1, 3))
bob_physics[9] = Active

=== build_alice_q: Alice physics CNF clauses ===
alice_physics[0] = At(0, 0)
  cnf[0] (from local 0, weight=100.0) = At(0, 0)
alice_physics[1] = At(2, 3)
  cnf[1] (from local 0, weight=100.0) = At(2, 3)
Total CNF clauses: 2

Alice start/goal: 0 -> 3
Q_alice shape: (2, 72)
w_alice shape: (2,)
Q_alice rows:
[[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]
Alice clause weights: [100. 100.]


=== build_physics_q: Physics CNF clauses ===
physics[0] = (((At(0, 0) | At(0, 1)) | At(0, 2)) | At(0, 3))
  cnf[0] (from local 0, weight=100.0) = (At(0, 0) | At(0, 1) | At(0, 2) | At(0, 3))
physics[1] = (~At(0, 0) | ~At(0, 1))
  cnf[1] (from local 0, weight=100.0) = (~At(0, 0) | ~At(0, 1))
physics[2] = (~At(0, 0) | ~At(0, 2))
  cnf[2] (from local 0, weight=100.0) = (~At(0, 0) | ~At(0, 2))
physics[3] = (~At(0, 0) | ~At(0, 3))
  cnf[3] (from local 0, weight=100.0) = (~At(0, 0) | ~At(0, 3))
physics[4] = (~At(0, 1) | ~At(0, 2))
  cnf[4] (from local 0, weight=100.0) = (~At(0, 1) | ~At(0, 2))
physics[5] = (~At(0, 1) | ~At(0, 3))
  cnf[5] (from local 0, weight=100.0) = (~At(0, 1) | ~At(0, 3))
physics[6] = (~At(0, 2) | ~At(0, 3))
  cnf[6] (from local 0, weight=100.0) = (~At(0, 2) | ~At(0, 3))
physics[7] = (((At(1, 0) | At(1, 1)) | At(1, 2)) | At(1, 3))
  cnf[7] (from local 0, weight=100.0) = (At(1, 0) | At(1, 1) | At(1, 2) | At(1, 3))
physics[8] = (~At(1, 0) | ~At(1, 1))
  cnf[8] (from local 0